# H1 · Risk and stability: do stronger institutions mean fewer loss-making firms?

> **Hypothesis H1:** Where institutions are stronger, fewer listed companies make a loss and profits are less uneven.

| | |
|---|---|
| **Why we ask** | A first scan of many outcomes showed the **share of loss-making company-years** had more links to country features (13 of 35) than any other outcome, including average ROA. Risk may matter more than the average. |
| **Prediction** | **Negative** association between the institutions index and the loss share. |
| **Data** | Yahoo company-years (10,220). *Institutions index* = average of the six governance indicators (rule of law, corruption control, government effectiveness, regulatory quality, voice, political stability), previous year, in SDs across countries. ESEF for replication. |
| **Primary test (fixed before looking at the result)** | `loss_pp ~ institutions index + sector + size rank + fiscal year`, where `loss_pp` = 100 if the company's net income is negative and 0 otherwise. Errors clustered by country; wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · What "risk" looks like in the data
Four country-level views of risk, computed from the firms in each country:

| Measure | Meaning |
|---|---|
| **Loss share** | % of company-years with negative net income |
| **ROA spread** | gap between the 75th and 25th percentile ROA: how uneven profits are |
| **ROA 10th percentile** | what a bad (but not worst) year looks like |
| **Firm ROA volatility** | typical year-to-year swing of one company's ROA (companies with 3+ years) |

In [2]:
g = fy.groupby("iso3")
vol = fy.groupby("company_id").agg(n=("roa_w", "size"), sd=("roa_w", "std"), iso3=("iso3", "first")).query("n >= 3").groupby("iso3").sd.median()
risk = pd.DataFrame({"loss share (%)": g.loss_pp.mean(), "ROA spread (IQR, pp)": (g.roa_w.quantile(.75) - g.roa_w.quantile(.25)) * 100,
                     "ROA 10th pct (pp)": g.roa_w.quantile(.10) * 100, "firm ROA volatility (pp)": vol * 100, "firms": g.company_id.nunique()})
risk = (risk[risk.firms >= MIN_FIRMS].join(country["inst_idx"].rename("institutions index")).join(country["dev_idx"].rename("development index"))
        .join(dim[["name", "region"]]))
print(f"{len(risk)} countries. Loss share across countries: min {risk['loss share (%)'].min():.1f}%, median {risk['loss share (%)'].median():.1f}%, max {risk['loss share (%)'].max():.1f}%")
risk.sort_values("loss share (%)")[["name", "loss share (%)", "ROA spread (IQR, pp)", "ROA 10th pct (pp)", "firm ROA volatility (pp)", "institutions index", "firms"]].round(2)

34 countries. Loss share across countries: min 3.3%, median 14.5%, max 30.7%


,name,loss share (%),"ROA spread (IQR, pp)",ROA 10th pct (pp),firm ROA volatility (pp),institutions index,firms
iso3,,,,,,,
JPN,Japan,3.260,4.670,0.560,1.090,0.780,100
IND,India,3.320,8.470,1.410,1.280,-1.610,100
CHN,China,6.250,6.890,0.460,0.970,-1.880,100
USA,United States,6.280,9.850,0.900,2.350,0.050,100
CZE,Czechia,8.820,6.880,0.810,2.120,0.200,17
GRC,Greece,10.100,6.450,-0.000,1.730,-0.670,81
ZAF,South Africa,10.200,8.050,-0.030,2.140,-1.380,100
IDN,Indonesia,10.550,7.280,-0.350,1.270,-1.540,100
BRA,Brazil,11.480,6.290,-1.310,1.920,-1.750,99


In [3]:
meas = ["loss share (%)", "ROA spread (IQR, pp)", "ROA 10th pct (pp)", "firm ROA volatility (pp)"]
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{m} (ρ = {spearman(risk['institutions index'], risk[m])[0]:+.2f}, n = {len(risk)})" for m in meas], vertical_spacing=0.16)
for k, m in enumerate(meas):
    fig.add_trace(go.Scatter(x=risk["institutions index"], y=risk[m], mode="markers+text", text=risk.index, textposition="top center", textfont_size=8,
                             marker=dict(size=7, color="#4C78A8"), showlegend=False), row=k // 2 + 1, col=k % 2 + 1)
    d = risk[["institutions index", m]].dropna()
    b, a = np.polyfit(d["institutions index"], d[m], 1); xs = np.array([d["institutions index"].min(), d["institutions index"].max()])
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#E45756"), showlegend=False), row=k // 2 + 1, col=k % 2 + 1)
fig.update_xaxes(title_text="institutions index (SD)", row=2); fig.update_layout(height=720, title="Risk measures vs institutions index, one point per country")
fig.show()

## 2 · Primary test (firm level)
Each company-year gets a 0/1 loss flag. The model compares companies **in the same sector, of similar relative size and in the same fiscal year**; the coefficient is the change in the percentage-point probability of a loss for a country with a +1 SD institutions index. Errors are clustered by country and the p-value comes from a wild cluster bootstrap (appropriate with 34 countries).

In [4]:
primary = wild_cluster("inst_z", fy, outcome="loss_pp", B=1999)
base_rate = fy.loss_pp.mean()
print(f"Base rate of loss-making company-years: {base_rate:.1f}%")
print(f"+1 SD institutions → {primary['coef']:+.2f} pp probability of a loss (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}); "
      f"that is {primary['coef'] / base_rate:+.0%} of the base rate.")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f} | {primary['clusters']} countries, {primary['n']:,} company-years")

Base rate of loss-making company-years: 14.6%
+1 SD institutions → +3.11 pp probability of a loss (95% CI +1.22 to +5.01); that is +21% of the base rate.
p (cluster-robust) = 0.0029 | p (wild bootstrap) = 0.0135 | 34 countries, 10,220 company-years


### Secondary outcomes: how uneven are profits?
Same model, other outcomes. *Deviation from the norm* is how far a company's ROA is from the median ROA of its sector and fiscal year, in absolute pp.

In [5]:
fy["abs_dev_pp"] = (fy.roa_pp - fy.groupby(["sector", "fiscal_year"]).roa_pp.transform("median")).abs()
sec = {"loss (pp probability)": "loss_pp", "deviation of ROA from the sector norm (pp)": "abs_dev_pp", "ROA level (pp), for context": "roa_pp"}
rows = []
for lab, col in sec.items():
    r = wild_cluster("inst_z", fy, outcome=col, B=999)
    rows.append({"outcome": lab, "coef per +1 SD": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"]})
secondary = pd.DataFrame(rows).set_index("outcome")
secondary

,coef per +1 SD,ci_low,ci_high,p (wild)
outcome,,,,
loss (pp probability),3.114,1.219,5.009,0.012
deviation of ROA from the sector norm (pp),0.219,-0.202,0.639,0.375
"ROA level (pp), for context",-1.016,-1.497,-0.536,0.001


## 3 · What actually goes with loss-making?
The hypothesis names institutions, but the first scan pointed at other features too. This table checks all 35 country features against the country loss share (n = 34, Spearman, Benjamini–Hochberg q across the 35 tests), so we can see whether institutions are really the main story or just one member of a correlated block.

In [6]:
ft = profile.loc[risk.index]
s = scan(ft, risk["loss share (%)"], feat_cols)
print(f"{(s.p < 0.05).sum()} of {len(s)} features have p < 0.05 (about {0.05 * len(s):.1f} by chance); {(s.q < 0.10).sum()} have q < 0.10")
s.head(10)

13 of 35 features have p < 0.05 (about 1.8 by chance); 10 have q < 0.10


,rho,p,n,q
feature,,,,
trade_gdp,0.670,0.000,34,0.001
population,-0.635,0.000,34,0.001
gini,-0.575,0.002,26,0.025
internet_users,0.459,0.006,34,0.045
gdp_usd,-0.458,0.006,34,0.045
corruption_control,0.444,0.009,34,0.049
regulatory_quality,0.437,0.010,34,0.049
voice_accountability,0.426,0.012,34,0.053
rule_of_law,0.396,0.020,34,0.078


**Do institutions matter beyond openness and size?** Country-level regression of loss share on the institutions index, controlling for trade openness and population (n = 34, robust errors). If the institutions coefficient vanishes, the pattern is really about openness and size.

In [7]:
cd = risk.join(profile[["trade_gdp", "population"]])
for c in ["trade_gdp", "population"]:
    cd[c + "_z"] = (cd[c] - cd[c].mean()) / cd[c].std()
cd = cd.rename(columns={"loss share (%)": "loss", "institutions index": "inst"})
mods = {"institutions only": "loss ~ inst", "+ trade openness": "loss ~ inst + trade_gdp_z", "+ trade openness + population": "loss ~ inst + trade_gdp_z + population_z"}
rows = []
for lab, f in mods.items():
    m = smf.ols(f, cd).fit(cov_type="HC3")
    rows.append({"model": lab, "institutions coef (pp loss share per +1 SD)": m.params["inst"], "p": m.pvalues["inst"], "R²": m.rsquared,
                 "trade coef": m.params.get("trade_gdp_z", np.nan), "population coef": m.params.get("population_z", np.nan)})
pd.DataFrame(rows).set_index("model")

,institutions coef (pp loss share per +1 SD),p,R²,trade coef,population coef
model,,,,,
institutions only,2.682,0.007,0.165,NaN,NaN
+ trade openness,1.947,0.108,0.216,1.671,NaN
+ trade openness + population,0.117,0.934,0.390,-0.741,-4.543


## 4 · Robustness
The primary test again, in seven samples (wild bootstrap with 499 draws each), plus a variant that also controls for the development index.

In [8]:
rob, rob_share = robustness("inst_z", "loss_pp", -1)
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low,
                 title="Institutions → loss probability (pp per +1 SD) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
extra = wild_cluster("inst_z", fy, outcome="loss_pp", extra="dev_z", B=499)
print(f"Expected sign (negative) in {rob_share:.0%} of samples. Also controlling for the development index: {extra['coef']:+.2f} pp (p wild = {extra['p_wild']:.3f}).")
rob

Expected sign (negative) in 0% of samples. Also controlling for the development index: +2.46 pp (p wild = 0.472).


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,3.114,1.219,5.009,0.010,34,10220,False
excluding financials & real estate,2.861,0.585,5.137,0.026,34,8063,False
excluding small-sample countries,3.072,1.116,5.028,0.012,29,9787,False
"excluding USA, China, Japan",3.483,1.695,5.272,0.004,31,9023,False
fiscal years 2023 and later,2.946,0.859,5.033,0.014,34,7603,False
EU countries only,4.108,1.116,7.099,0.034,21,5121,False
non-EU countries only,0.831,-1.373,3.035,0.528,13,5099,False


## 5 · Replication on ESEF
ESEF is a different source (EU annual reports, IFRS, no sector field, so only size and year are controlled). It covers 20 EU countries. It overlaps with Yahoo for some companies, so it is a **partly independent** check.

In [9]:
fe = prep.load_firms(con, country, source="ESEF")
rep = wild_cluster("inst_z", fe, outcome="loss_pp", controls="size_rank + C(fiscal_year)", B=1999)
ge = fe.groupby("iso3").agg(loss=("loss_pp", "mean"), firms=("company_id", "nunique")).query("firms >= 10").join(country["inst_idx"].rename("inst"))
rho_e, p_e, n_e = spearman(ge.inst, ge.loss)
print(f"ESEF, firm level ({rep['n']:,} company-years, {rep['clusters']} countries): +1 SD institutions → {rep['coef']:+.2f} pp loss probability "
      f"(95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")
print(f"ESEF, country level: Spearman ρ between loss share and institutions = {rho_e:+.2f} (p = {p_e:.3f}, n = {n_e})")

ESEF, firm level (5,312 company-years, 19 countries): +1 SD institutions → +4.85 pp loss probability (95% CI +2.41 to +7.29), wild p = 0.013
ESEF, country level: Spearman ρ between loss share and institutions = +0.73 (p = 0.000, n = 19)


## 6 · Verdict

In [10]:
v = utils.verdict(primary["coef"], primary["p_wild"], -1, rob_share, primary["clusters"])
rep_same = bool(np.sign(rep["coef"]) == -1)
explained_away = bool(pd.DataFrame(rows).iloc[-1, 2] > 0.10)
lines = [
 f"Primary: +1 SD institutions → {primary['coef']:+.2f} pp probability of a loss (base rate {base_rate:.1f}%), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries.",
 f"Robustness: expected sign in {rob_share:.0%} of seven samples. Replication on ESEF: {rep['coef']:+.2f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"Beyond openness and size: institutions coefficient {pd.DataFrame(rows).iloc[-1, 1]:+.2f} (p = {pd.DataFrame(rows).iloc[-1, 2]:.2f}) at country level when trade openness and population are controlled.",
 f"VERDICT H1: {v.upper()}" + (" (the association is explained away by trade openness and population)" if explained_away else ""),
]
print("\n".join(lines))
utils.save_result("H1", id="H1", title="Risk and stability", statement="Where institutions are stronger, fewer listed companies make a loss.",
                  expected_sign=-1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp loss probability per +1 SD institutions",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"],
                  robust_share=rob_share, replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same,
                  verdict=v, explained_away=explained_away, notebook="1.1_risk_and_stability.ipynb", notes=lines)

Primary: +1 SD institutions → +3.11 pp probability of a loss (base rate 14.6%), wild p = 0.013, 34 countries.
Robustness: expected sign in 0% of seven samples. Replication on ESEF: +4.85 pp (wild p = 0.013), opposite direction.
Beyond openness and size: institutions coefficient +0.12 (p = 0.93) at country level when trade openness and population are controlled.
VERDICT H1: CONTRADICTED (the association is explained away by trade openness and population)


### Limits
- The institutions index and "development" are strongly correlated across the 34 countries, so this test cannot cleanly separate institutions from wealth.
- A loss in one year can be a one-off (write-down, restructuring); the loss flag is a rough measure of risk, not a full risk model.
- The ESEF replication shares some companies with the Yahoo sample and has no sector control.
- Association, not causation.

In [11]:
con.close()